# BirdCLEF+ 2026 — Week 2+3 Combined Training

## What this notebook does

| Phase | Content | Est. Time |
|---|---|---|
| Phase 1 | 5-fold training, on-the-fly loading, random offset | ~13 hours |
| Phase 2 | Pseudo-label generation on train_soundscapes | ~30 min |
| Phase 3 | Fine-tune 2 folds with pseudo-labels | ~2.5 hours |
| Upload | Auto-push all models to Kaggle Dataset | ~5 min |
| **Total** | | **~17 hours** |

## Key fixes vs previous runs
- **No spec_cache** — on-the-fly loading with random offset gives Week 1 quality (OOF 0.96)
- **BCE loss** — stable, proven from Week 1
- **Random offset** — model sees different 5s windows each epoch, critical for diversity
- **Auto-upload** — models pushed to Kaggle Dataset while internet is ON

**Settings: Internet ON · GPU T4 · ~17h runtime**

In [ ]:
import os, gc, math, random, warnings, json, shutil, subprocess, time as _time
from pathlib import Path
from typing import List

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import soundfile as sf
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import timm
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

seed_everything(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch :', torch.__version__)
print('Device  :', DEVICE)
if torch.cuda.is_available():
    print('GPU     :', torch.cuda.get_device_name(0))

In [ ]:
class CFG:
    # ── Paths ──────────────────────────────────────────────────────────────
    BASE_DIR         = Path('/kaggle/input/competitions/birdclef-2026')
    TRAIN_AUDIO_DIR  = BASE_DIR / 'train_audio'
    SOUNDSCAPE_DIR   = BASE_DIR / 'train_soundscapes'
    TRAIN_CSV        = BASE_DIR / 'train.csv'
    SAMPLE_SUB       = BASE_DIR / 'sample_submission.csv'
    OUTPUT_DIR       = Path('/kaggle/working')

    # ── Audio ──────────────────────────────────────────────────────────────
    SAMPLE_RATE      = 32000
    WINDOW_SIZE      = 5

    # ── Mel spectrogram — identical to Week 1 and inference notebook ───────
    N_FFT            = 1024
    HOP_LENGTH       = 64
    N_MELS           = 136
    FMIN             = 20
    FMAX             = 16000
    TARGET_SHAPE     = (256, 256)

    # ── Model ──────────────────────────────────────────────────────────────
    MODEL_NAME       = 'efficientnet_b0'
    WEIGHTS_PATH     = None  # None = download via timm (internet ON)

    # ── Phase 1: 5-fold training ───────────────────────────────────────────
    N_FOLDS          = 5
    TRAIN_FOLDS      = [0, 1, 2, 3, 4]
    EPOCHS           = 10
    BATCH_SIZE       = 64
    NUM_WORKERS      = 0    # 0 = no multiprocessing crash spam
    LR               = 1e-3
    WEIGHT_DECAY     = 1e-4
    WARMUP_EPOCHS    = 1
    MIN_LR           = 1e-6
    MIXUP_ALPHA      = 0.15

    # ── SpecAugment ────────────────────────────────────────────────────────
    FREQ_MASK_MAX    = 30
    TIME_MASK_MAX    = 60
    N_FREQ_MASKS     = 2
    N_TIME_MASKS     = 2

    # ── Phase 2: Pseudo-labeling ───────────────────────────────────────────
    PL_CONFIDENCE    = 0.5   # min prediction confidence to accept as pseudo-label
    PL_FINETUNE_FOLDS = [0, 1]  # folds to fine-tune with pseudo-labels
    PL_FINETUNE_EPOCHS = 5      # additional epochs with pseudo-labels

    # ── Kaggle Dataset auto-upload ─────────────────────────────────────────
    DATASET_NAME     = 'birdclef2026-week23-models'

    SEED             = 42
    DEBUG            = False

cfg = CFG()
cfg.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Train audio exists    :', cfg.TRAIN_AUDIO_DIR.exists())
print('Soundscape dir exists :', cfg.SOUNDSCAPE_DIR.exists())
print('Output dir            :', cfg.OUTPUT_DIR)

In [ ]:
# ── Load training metadata ─────────────────────────────────────────────────
train_df = pd.read_csv(cfg.TRAIN_CSV)
species_col = 'primary_label' if 'primary_label' in train_df.columns else 'species_code'
print(f'Train rows  : {len(train_df)} | species col: "{species_col}"')

all_classes = sorted(train_df[species_col].unique())
NUM_CLASSES = len(all_classes)
le = LabelEncoder()
le.fit(all_classes)
class_list = le.classes_.tolist()
print(f'Classes     : {NUM_CLASSES}')

train_df['label_idx'] = le.transform(train_df[species_col])

if cfg.DEBUG:
    train_df = train_df.groupby(species_col).head(2).reset_index(drop=True)
    print(f'DEBUG: {len(train_df)} rows')

# Class imbalance check
counts = train_df[species_col].value_counts()
print(f'Min/Median/Max samples: {counts.min()} / {counts.median():.0f} / {counts.max()}')

In [ ]:
# ── Audio pipeline ─────────────────────────────────────────────────────────

def load_audio(filepath, sr, offset=0.0, duration=5.0):
    """Load audio. Tile short clips instead of zero-padding."""
    target = int(sr * duration)
    try:
        y, _ = librosa.load(filepath, sr=sr, offset=offset,
                             duration=duration, mono=True)
    except Exception:
        return np.zeros(target, dtype=np.float32)
    if len(y) == 0:
        return np.zeros(target, dtype=np.float32)
    if len(y) < target:
        y = np.tile(y, math.ceil(target / len(y)))
    return y[:target].astype(np.float32)


def to_melspec(y, cfg):
    mel = librosa.feature.melspectrogram(
        y=y, sr=cfg.SAMPLE_RATE, n_fft=cfg.N_FFT,
        hop_length=cfg.HOP_LENGTH, n_mels=cfg.N_MELS,
        fmin=cfg.FMIN, fmax=cfg.FMAX, power=2.0)
    mel = librosa.power_to_db(mel, ref=np.max)
    mel = (mel - mel.min()) / (mel.max() - mel.min() + 1e-6)
    return mel.astype(np.float32)


def spec_augment(spec, cfg):
    spec = spec.copy()
    H, W = spec.shape
    for _ in range(cfg.N_FREQ_MASKS):
        f  = random.randint(0, cfg.FREQ_MASK_MAX)
        f0 = random.randint(0, max(0, H - f))
        spec[f0:f0+f, :] = 0.0
    for _ in range(cfg.N_TIME_MASKS):
        t  = random.randint(0, cfg.TIME_MASK_MAX)
        t0 = random.randint(0, max(0, W - t))
        spec[:, t0:t0+t] = 0.0
    return spec


def audio_to_tensor(filepath, cfg, offset=0.0, augment=False):
    """Full pipeline: filepath → (1, H, W) tensor."""
    y    = load_audio(filepath, cfg.SAMPLE_RATE, offset, cfg.WINDOW_SIZE)
    spec = to_melspec(y, cfg)
    if augment:
        spec = spec_augment(spec, cfg)
    spec = cv2.resize(spec, (cfg.TARGET_SHAPE[1], cfg.TARGET_SHAPE[0]),
                      interpolation=cv2.INTER_CUBIC)
    return torch.tensor(spec, dtype=torch.float32).unsqueeze(0)


def mixup(x, y, alpha=0.15):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1-lam) * x[idx], y, y[idx], lam


def mixup_loss(criterion, pred, ya, yb, lam):
    return lam * criterion(pred, ya) + (1-lam) * criterion(pred, yb)


print('Audio pipeline defined.')

In [ ]:
class BirdDataset(Dataset):
    """
    On-the-fly audio loading with random offset each epoch.
    This is the Week 1 approach that gave OOF AUC 0.96.
    Critical: random offset means the model sees different 5s
    windows of each clip every epoch — rich temporal diversity.
    """
    def __init__(self, df, cfg, augment=True, audio_dir=None):
        self.df       = df.reset_index(drop=True)
        self.cfg      = cfg
        self.augment  = augment
        self.audio_dir = audio_dir or cfg.TRAIN_AUDIO_DIR

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        fp  = str(self.audio_dir / row['filename'])

        # Random offset — different window every epoch (CRITICAL for quality)
        if self.augment:
            try:
                total   = sf.info(fp).duration
                max_off = max(0.0, total - self.cfg.WINDOW_SIZE)
                offset  = random.uniform(0, max_off) if max_off > 0 else 0.0
            except Exception:
                offset = 0.0
        else:
            offset = float(row.get('_offset', 0.0))

        tensor = audio_to_tensor(fp, self.cfg, offset, self.augment)

        # Label vector
        label = torch.zeros(NUM_CLASSES, dtype=torch.float32)
        label[int(row['label_idx'])] = 1.0

        # Secondary labels at 0.5 weight
        sec = str(row.get('secondary_labels', ''))
        if sec and sec not in ('nan', 'None', ''):
            for sp in sec.replace(',', ' ').split():
                sp = sp.strip()
                if sp in le.classes_:
                    label[le.transform([sp])[0]] = 0.5

        return tensor, label


print('BirdDataset defined (on-the-fly, random offset).')

In [ ]:
class GeM(nn.Module):
    def __init__(self, p=3.0, eps=1e-6):
        super().__init__()
        self.p   = nn.Parameter(torch.ones(1) * p)
        self.eps = eps
    def forward(self, x):
        return F.adaptive_avg_pool2d(
            x.clamp(self.eps).pow(self.p), 1).pow(1/self.p)


class BirdCLEFModel(nn.Module):
    def __init__(self, num_classes, cfg):
        super().__init__()
        if cfg.WEIGHTS_PATH and Path(cfg.WEIGHTS_PATH).exists():
            self.backbone = timm.create_model(
                cfg.MODEL_NAME, pretrained=False,
                num_classes=0, global_pool='', in_chans=1)
            sd  = torch.load(cfg.WEIGHTS_PATH, map_location='cpu', weights_only=True)
            key = 'conv_stem.weight'
            if key in sd:
                sd[key] = sd[key].mean(dim=1, keepdim=True)
            self.backbone.load_state_dict(sd, strict=False)
            print(f'Weights from {cfg.WEIGHTS_PATH}')
        else:
            self.backbone = timm.create_model(
                cfg.MODEL_NAME, pretrained=True,
                num_classes=0, global_pool='', in_chans=1)
            print('Pretrained weights via timm.')
        d = self.backbone.num_features
        self.pool = GeM()
        self.bn   = nn.BatchNorm1d(d)
        self.drop = nn.Dropout(0.3)
        self.fc   = nn.Linear(d, num_classes)

    def forward(self, x):
        x = self.backbone(x)
        x = self.pool(x).flatten(1)
        x = self.bn(x)
        x = self.drop(x)
        return self.fc(x)


# Quick test
_m = BirdCLEFModel(NUM_CLASSES, cfg).to(DEVICE)
_o = _m(torch.randn(2, 1, 256, 256).to(DEVICE))
print(f'Output shape : {_o.shape}')
print(f'Parameters   : {sum(p.numel() for p in _m.parameters())/1e6:.1f}M')
del _m, _o; gc.collect(); torch.cuda.empty_cache()

In [ ]:
def get_scheduler(optimizer, cfg, steps_per_epoch):
    total  = cfg.EPOCHS * steps_per_epoch
    warmup = cfg.WARMUP_EPOCHS * steps_per_epoch
    def lr_fn(step):
        if step < warmup:
            return step / max(1, warmup)
        prog = (step - warmup) / max(1, total - warmup)
        return max(cfg.MIN_LR / cfg.LR,
                   0.5 * (1 + math.cos(math.pi * prog)))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_fn)


def compute_auc(y_true, y_pred):
    aucs = []
    for i in range(y_true.shape[1]):
        if y_true[:, i].sum() > 0:
            try: aucs.append(roc_auc_score(y_true[:, i], y_pred[:, i]))
            except: pass
    return float(np.mean(aucs)) if aucs else 0.0


def train_epoch(model, loader, optimizer, scheduler, criterion, cfg):
    model.train()
    losses = []
    for x, y in tqdm(loader, desc='  Train', leave=False):
        x, y = x.to(DEVICE), y.to(DEVICE)
        if cfg.MIXUP_ALPHA > 0 and random.random() > 0.5:
            x, ya, yb, lam = mixup(x, y, cfg.MIXUP_ALPHA)
            loss = mixup_loss(criterion, model(x), ya, yb, lam)
        else:
            loss = criterion(model(x), y)
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        losses.append(loss.item())
    return np.mean(losses)


@torch.no_grad()
def validate(model, loader):
    model.eval()
    preds, labels = [], []
    for x, y in tqdm(loader, desc='  Valid', leave=False):
        preds.append(torch.sigmoid(model(x.to(DEVICE))).cpu().numpy())
        labels.append(y.numpy())
    preds  = np.concatenate(preds)
    labels = np.concatenate(labels)
    auc    = compute_auc((labels > 0.5).astype(int), preds)
    return auc, preds, labels


print('Training utilities defined.')

In [ ]:
# ── Stratified K-Fold split ────────────────────────────────────────────────
skf = StratifiedKFold(n_splits=cfg.N_FOLDS, shuffle=True, random_state=cfg.SEED)
train_df['fold'] = -1
for fold, (_, val_idx) in enumerate(
        skf.split(train_df, train_df['label_idx'])):
    train_df.loc[train_df.index[val_idx], 'fold'] = fold

print('Fold distribution:')
print(train_df['fold'].value_counts().sort_index().to_string())

## Phase 1 — 5-Fold Training

On-the-fly loading with random offset. Identical to Week 1 which gave OOF AUC 0.96.
Expected: ~13 hours total for 5 folds × 10 epochs.

In [ ]:
PHASE1_START = _time.time()

oof_preds  = np.zeros((len(train_df), NUM_CLASSES), dtype=np.float32)
oof_labels = np.zeros((len(train_df), NUM_CLASSES), dtype=np.float32)
best_aucs  = {}

criterion = nn.BCEWithLogitsLoss()

for fold in cfg.TRAIN_FOLDS:
    fold_start = _time.time()
    print(f'\n{"="*55}')
    print(f'  FOLD {fold}  |  '
          f'Elapsed so far: {(_time.time()-PHASE1_START)/3600:.1f}h')
    print(f'{"="*55}')

    trn_df = train_df[train_df['fold'] != fold].reset_index(drop=True)
    val_df = train_df[train_df['fold'] == fold].reset_index(drop=True)
    print(f'  Train: {len(trn_df)}  |  Val: {len(val_df)}')

    trn_ds = BirdDataset(trn_df, cfg, augment=True)
    val_ds = BirdDataset(val_df, cfg, augment=False)

    trn_loader = DataLoader(
        trn_ds, batch_size=cfg.BATCH_SIZE, shuffle=True,
        num_workers=cfg.NUM_WORKERS, pin_memory=True, drop_last=True)
    val_loader = DataLoader(
        val_ds, batch_size=cfg.BATCH_SIZE, shuffle=False,
        num_workers=cfg.NUM_WORKERS, pin_memory=True)

    model     = BirdCLEFModel(NUM_CLASSES, cfg).to(DEVICE)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)
    scheduler = get_scheduler(optimizer, cfg, len(trn_loader))

    best_auc  = 0.0
    best_path = cfg.OUTPUT_DIR / f'model_fold{fold}.pt'
    history   = []

    for epoch in range(1, cfg.EPOCHS + 1):
        trn_loss = train_epoch(
            model, trn_loader, optimizer, scheduler, criterion, cfg)
        val_auc, preds, labels = validate(model, val_loader)
        history.append({'epoch': epoch, 'loss': trn_loss, 'val_auc': val_auc})

        lr_now = scheduler.get_last_lr()[0]
        marker = ' ✔' if val_auc > best_auc else ''
        print(f'  E{epoch:02d}  '
              f'loss={trn_loss:.4f}  '
              f'auc={val_auc:.4f}  '
              f'lr={lr_now:.2e}'
              f'{marker}')

        if val_auc > best_auc:
            best_auc = val_auc
            torch.save(model.state_dict(), best_path)

    best_aucs[fold] = best_auc
    fold_hours = (_time.time() - fold_start) / 3600
    print(f'\n  Fold {fold} best AUC: {best_auc:.4f}  '
          f'({fold_hours:.1f}h)')

    # OOF predictions with best model
    model.load_state_dict(torch.load(best_path, map_location=DEVICE))
    val_idx_orig = train_df[train_df['fold'] == fold].index
    _, oof_p, oof_l = validate(model, val_loader)
    oof_preds[val_idx_orig]  = oof_p
    oof_labels[val_idx_orig] = (oof_l > 0.5).astype(np.float32)

    # Plot
    hist = pd.DataFrame(history)
    fig, ax1 = plt.subplots(figsize=(9, 3))
    ax1.plot(hist['epoch'], hist['loss'], 'b-o', ms=4)
    ax1.set_ylabel('BCE Loss', color='b')
    ax2 = ax1.twinx()
    ax2.plot(hist['epoch'], hist['val_auc'], 'r-s', ms=4)
    ax2.set_ylabel('Val AUC', color='r')
    ax1.set_xlabel('Epoch')
    plt.title(f'Fold {fold}  —  best AUC {best_auc:.4f}')
    plt.tight_layout()
    plt.savefig(cfg.OUTPUT_DIR / f'curve_fold{fold}.png', dpi=100)
    plt.show()

    del model, trn_loader, val_loader, trn_ds, val_ds
    gc.collect()
    torch.cuda.empty_cache()

print(f'\n{"="*55}')
print(f'Phase 1 complete in {(_time.time()-PHASE1_START)/3600:.1f}h')
print('Best AUCs:', {k: round(v, 4) for k, v in best_aucs.items()})
print(f'Mean AUC : {np.mean(list(best_aucs.values())):.4f}')

In [ ]:
# OOF Summary
oof_auc = compute_auc(oof_labels, oof_preds)
print(f'Overall OOF AUC (all folds): {oof_auc:.4f}')

per_class = []
for i in range(NUM_CLASSES):
    t, p = oof_labels[:, i], oof_preds[:, i]
    if t.sum() > 0:
        try: per_class.append(roc_auc_score(t, p))
        except: pass

plt.figure(figsize=(8, 3))
plt.hist(per_class, bins=30, color='steelblue', edgecolor='white')
plt.axvline(np.mean(per_class), color='red', linestyle='--',
            label=f'Mean = {np.mean(per_class):.3f}')
plt.xlabel('Per-class ROC-AUC')
plt.title('OOF per-class AUC (all 5 folds)')
plt.legend(); plt.tight_layout()
plt.savefig(cfg.OUTPUT_DIR / 'oof_auc.png', dpi=100)
plt.show()
print(f'Classes < 0.7 AUC : {sum(a < 0.7 for a in per_class)}')
print(f'Classes < 0.8 AUC : {sum(a < 0.8 for a in per_class)}')

# List saved models
print('\nSaved models:')
for f in sorted(cfg.OUTPUT_DIR.glob('model_fold*.pt')):
    print(f'  {f.name}  ({f.stat().st_size/1e6:.0f} MB)')

## Phase 2 — Pseudo-Label Generation (Week 3)

Run the 5-model ensemble on `train_soundscapes` to generate pseudo-labels.
These are the same domain as the test set — adding them to training directly closes the domain gap.

Only windows where the model is confident (max prediction > threshold) are kept.

In [ ]:
# ── Find soundscape files ──────────────────────────────────────────────────
soundscape_files = sorted(cfg.SOUNDSCAPE_DIR.glob('*.ogg'))
if not soundscape_files:
    soundscape_files = sorted(cfg.SOUNDSCAPE_DIR.glob('*.wav'))
print(f'Soundscape files found: {len(soundscape_files)}')

if len(soundscape_files) == 0:
    print('No soundscape files — skipping pseudo-labeling.')
    pseudo_df = pd.DataFrame()
else:
    print(f'First 3: {[f.name for f in soundscape_files[:3]]}')

In [ ]:
if len(soundscape_files) > 0:

    class SoundscapeDataset(Dataset):
        """Split each soundscape into 5s non-overlapping windows."""
        def __init__(self, filepaths, cfg):
            self.cfg     = cfg
            self.samples = []
            for fp in filepaths:
                stem = Path(fp).stem
                try:
                    dur = sf.info(fp).duration
                except Exception:
                    dur = 60.0
                for off in np.arange(0, dur, cfg.WINDOW_SIZE):
                    self.samples.append((str(fp), float(off), stem))
            print(f'{len(filepaths)} soundscapes → '
                  f'{len(self.samples)} windows')

        def __len__(self):
            return len(self.samples)

        def __getitem__(self, idx):
            fp, off, stem = self.samples[idx]
            tensor = audio_to_tensor(fp, self.cfg, off, augment=False)
            return tensor, off, stem


    # Load all 5 fold models
    pl_models = []
    for fold in cfg.TRAIN_FOLDS:
        mp = cfg.OUTPUT_DIR / f'model_fold{fold}.pt'
        if mp.exists():
            m = BirdCLEFModel(NUM_CLASSES, cfg).to(DEVICE)
            m.load_state_dict(
                torch.load(str(mp), map_location=DEVICE))
            m.eval()
            pl_models.append(m)
    print(f'Loaded {len(pl_models)} models for pseudo-labeling')

    # Run ensemble inference on soundscapes
    sl_dataset = SoundscapeDataset(soundscape_files, cfg)
    sl_loader  = DataLoader(
        sl_dataset, batch_size=32,
        shuffle=False, num_workers=0)

    all_preds   = []
    all_offsets = []
    all_stems   = []

    with torch.no_grad():
        for x, offsets, stems in tqdm(sl_loader, desc='Pseudo-label inference'):
            x = x.to(DEVICE)
            batch_pred = torch.zeros(x.size(0), NUM_CLASSES, device=DEVICE)
            for m in pl_models:
                batch_pred += torch.sigmoid(m(x)) / len(pl_models)
            all_preds.append(batch_pred.cpu().numpy())
            all_offsets.extend(offsets.tolist())
            all_stems.extend(stems)

    all_preds = np.concatenate(all_preds, axis=0)
    print(f'Predictions shape: {all_preds.shape}')

    # Free model memory
    for m in pl_models:
        del m
    gc.collect()
    torch.cuda.empty_cache()

    # Filter: keep windows with max confidence > threshold
    max_conf   = all_preds.max(axis=1)
    top_class  = all_preds.argmax(axis=1)
    keep_mask  = max_conf >= cfg.PL_CONFIDENCE

    print(f'Windows total    : {len(all_preds)}')
    print(f'Windows kept     : {keep_mask.sum()} '
          f'(conf >= {cfg.PL_CONFIDENCE})')
    print(f'Windows dropped  : {(~keep_mask).sum()}')

    # Build pseudo-label DataFrame in same format as train_df
    pl_rows = []
    for i in np.where(keep_mask)[0]:
        stem    = all_stems[i]
        offset  = all_offsets[i]
        pred_idx = top_class[i]
        sp_code  = class_list[pred_idx]
        pl_rows.append({
            'filename'  : f'{stem}.ogg',
            species_col : sp_code,
            'label_idx' : pred_idx,
            'fold'      : -1,          # assigned later
            '_offset'   : offset,
            '_is_pseudo': True,
        })

    pseudo_df = pd.DataFrame(pl_rows)
    print(f'\nPseudo-label rows: {len(pseudo_df)}')
    if len(pseudo_df) > 0:
        print(f'Unique species   : '
              f'{pseudo_df[species_col].nunique()}')
        pseudo_df.to_csv(
            cfg.OUTPUT_DIR / 'pseudo_labels.csv', index=False)
        print('Saved: pseudo_labels.csv')
else:
    pseudo_df = pd.DataFrame()
    print('Skipped pseudo-labeling (no soundscape files).')

## Phase 3 — Fine-tune 2 Folds with Pseudo-Labels

Take the best existing fold models and continue training for 5 more epochs
with pseudo-labeled soundscape windows added to the training set.

This directly addresses the domain gap between clean training clips and soundscape test data.

In [ ]:
if len(pseudo_df) > 0:
    PHASE3_START = _time.time()

    class PseudoDataset(Dataset):
        """Dataset that mixes clip data and pseudo-labeled soundscapes."""
        def __init__(self, clip_df, pl_df, cfg, augment=True):
            # Combine clip + pseudo-label rows
            pl_df = pl_df.copy()
            pl_df['secondary_labels'] = ''
            combined = pd.concat(
                [clip_df, pl_df], ignore_index=True)
            self.df      = combined.reset_index(drop=True)
            self.cfg     = cfg
            self.augment = augment
            n_clip = len(clip_df)
            n_pl   = len(pl_df)
            print(f'  PseudoDataset: {n_clip} clips + {n_pl} pseudo = '
                  f'{len(self.df)} total')

        def __len__(self):
            return len(self.df)

        def __getitem__(self, idx):
            row = self.df.iloc[idx]
            is_pseudo = bool(row.get('_is_pseudo', False))

            if is_pseudo:
                # Load from soundscape at fixed offset
                fp     = str(cfg.SOUNDSCAPE_DIR / row['filename'])
                offset = float(row.get('_offset', 0.0))
            else:
                # Load from train_audio with random offset
                fp = str(cfg.TRAIN_AUDIO_DIR / row['filename'])
                try:
                    total   = sf.info(fp).duration
                    max_off = max(0.0, total - self.cfg.WINDOW_SIZE)
                    offset  = (random.uniform(0, max_off)
                               if self.augment and max_off > 0 else 0.0)
                except Exception:
                    offset = 0.0

            tensor = audio_to_tensor(fp, self.cfg, offset, self.augment)

            label = torch.zeros(NUM_CLASSES, dtype=torch.float32)
            label[int(row['label_idx'])] = 1.0

            sec = str(row.get('secondary_labels', ''))
            if sec and sec not in ('nan', 'None', ''):
                for sp in sec.replace(',', ' ').split():
                    sp = sp.strip()
                    if sp in le.classes_:
                        label[le.transform([sp])[0]] = 0.5

            return tensor, label


    # Assign pseudo-label rows to folds round-robin
    pseudo_df = pseudo_df.reset_index(drop=True)
    pseudo_df['fold'] = pseudo_df.index % cfg.N_FOLDS

    # Fine-tune selected folds
    ft_criterion = nn.BCEWithLogitsLoss()

    for fold in cfg.PL_FINETUNE_FOLDS:
        print(f'\n{"="*55}')
        print(f'  FINE-TUNE FOLD {fold} (with pseudo-labels)')
        print(f'{"="*55}')

        # Load the best model from Phase 1
        base_path = cfg.OUTPUT_DIR / f'model_fold{fold}.pt'
        save_path = cfg.OUTPUT_DIR / f'model_fold{fold}_ft.pt'

        trn_clips = train_df[train_df['fold'] != fold].reset_index(drop=True)
        val_clips = train_df[train_df['fold'] == fold].reset_index(drop=True)
        trn_pl    = pseudo_df[pseudo_df['fold'] != fold].reset_index(drop=True)

        trn_ds = PseudoDataset(trn_clips, trn_pl, cfg, augment=True)
        val_ds = BirdDataset(val_clips, cfg, augment=False)

        trn_loader = DataLoader(
            trn_ds, batch_size=cfg.BATCH_SIZE, shuffle=True,
            num_workers=cfg.NUM_WORKERS, pin_memory=True, drop_last=True)
        val_loader = DataLoader(
            val_ds, batch_size=cfg.BATCH_SIZE, shuffle=False,
            num_workers=cfg.NUM_WORKERS, pin_memory=True)

        # Load pre-trained fold model and fine-tune at lower LR
        model = BirdCLEFModel(NUM_CLASSES, cfg).to(DEVICE)
        model.load_state_dict(
            torch.load(str(base_path), map_location=DEVICE))

        ft_cfg        = CFG()
        ft_cfg.EPOCHS = cfg.PL_FINETUNE_EPOCHS
        ft_cfg.LR     = cfg.LR * 0.1   # 10x lower LR for fine-tuning
        ft_cfg.WARMUP_EPOCHS = 0

        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=ft_cfg.LR,
            weight_decay=cfg.WEIGHT_DECAY)
        scheduler = get_scheduler(optimizer, ft_cfg, len(trn_loader))

        best_auc  = best_aucs.get(fold, 0.0)
        history   = []

        for epoch in range(1, ft_cfg.EPOCHS + 1):
            trn_loss = train_epoch(
                model, trn_loader, optimizer, scheduler,
                ft_criterion, ft_cfg)
            val_auc, _, _ = validate(model, val_loader)
            history.append({'epoch': epoch, 'loss': trn_loss,
                            'val_auc': val_auc})

            marker = ' ✔' if val_auc > best_auc else ''
            print(f'  FT E{epoch:02d}  '
                  f'loss={trn_loss:.4f}  '
                  f'auc={val_auc:.4f}'
                  f'{marker}')

            if val_auc > best_auc:
                best_auc = val_auc
                torch.save(model.state_dict(), save_path)
                print(f'  Saved: {save_path.name}')

        if not save_path.exists():
            # If fine-tuning didn't improve, copy the base model
            shutil.copy(base_path, save_path)
            print(f'  Fine-tuning did not improve — '
                  f'kept base model as {save_path.name}')

        best_aucs[f'{fold}_ft'] = best_auc
        print(f'\n  Fold {fold} fine-tune best AUC: {best_auc:.4f}')

        del model, trn_loader, val_loader, trn_ds, val_ds
        gc.collect()
        torch.cuda.empty_cache()

    print(f'\nPhase 3 complete in '
          f'{(_time.time()-PHASE3_START)/3600:.1f}h')

else:
    print('Skipping Phase 3 — no pseudo-labels available.')
    print('This is fine — 5-fold ensemble alone is a strong submission.')

# Final model inventory
print('\n── Final model files ──────────────────────────')
for f in sorted(cfg.OUTPUT_DIR.glob('model_fold*.pt')):
    print(f'  {f.name}  ({f.stat().st_size/1e6:.0f} MB)')

## Auto-Upload Models to Kaggle Dataset

Pushes all model files to a new Kaggle Dataset while internet is still ON.
No manual download needed.

In [ ]:
# ── Get Kaggle username ────────────────────────────────────────────────────
result = subprocess.run(
    ['kaggle', 'config', 'view'],
    capture_output=True, text=True)
print(result.stdout)

# Extract username from config
username = None
for line in result.stdout.splitlines():
    if 'username' in line.lower():
        username = line.split(':')[-1].strip()
        break

if not username:
    # Fallback: check environment variable
    username = os.environ.get('KAGGLE_USERNAME', '')

print(f'Username: {username}')
assert username, 'Could not detect Kaggle username. Check kaggle config.'

In [ ]:
# ── Create dataset folder with only model files ────────────────────────────
dataset_dir = Path('/kaggle/working/upload_dataset')
if dataset_dir.exists():
    shutil.rmtree(dataset_dir)
dataset_dir.mkdir()

# Copy all model .pt files
copied = []
for pt in sorted(cfg.OUTPUT_DIR.glob('model_fold*.pt')):
    shutil.copy(pt, dataset_dir / pt.name)
    copied.append(pt.name)
    print(f'Copied: {pt.name}  ({pt.stat().st_size/1e6:.0f} MB)')

print(f'\nTotal models: {len(copied)}')

# Write Kaggle dataset metadata
metadata = {
    'title': cfg.DATASET_NAME,
    'id': f'{username}/{cfg.DATASET_NAME}',
    'licenses': [{'name': 'CC0-1.0'}]
}
with open(dataset_dir / 'dataset-metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)
print(f'Metadata written for: {username}/{cfg.DATASET_NAME}')

In [ ]:
# ── Push to Kaggle ─────────────────────────────────────────────────────────
print('Uploading to Kaggle...')
result = subprocess.run(
    ['kaggle', 'datasets', 'create',
     '-p', str(dataset_dir),
     '--dir-mode', 'zip'],
    capture_output=True, text=True)

print('STDOUT:', result.stdout)
if result.stderr:
    print('STDERR:', result.stderr)

if result.returncode == 0:
    url = f'https://www.kaggle.com/datasets/{username}/{cfg.DATASET_NAME}'
    print(f'\nDataset created: {url}')
    print(f'\nUpdate inference notebook MODEL_PATHS to:')
    print('MODEL_PATHS = [')
    for name in copied:
        fold_name = name.replace('.pt', '')
        print(f"    '/kaggle/input/{cfg.DATASET_NAME}/{name}',")
    print(']')
else:
    print('Upload failed. Download files manually from Output tab.')
    print('Files to download:')
    for name in copied:
        print(f'  {name}')

## Update Inference Notebook

After this notebook completes, update your inference notebook with the new model paths printed above.

### Submission strategy

**Option A — 5-model ensemble (base):**
```python
MODEL_PATHS = [
    '/kaggle/input/birdclef2026-week23-models/model_fold0.pt',
    '/kaggle/input/birdclef2026-week23-models/model_fold1.pt',
    '/kaggle/input/birdclef2026-week23-models/model_fold2.pt',
    '/kaggle/input/birdclef2026-week23-models/model_fold3.pt',
    '/kaggle/input/birdclef2026-week23-models/model_fold4.pt',
]
```

**Option B — 7-model ensemble (base + fine-tuned folds 0 and 1):**
```python
MODEL_PATHS = [
    '/kaggle/input/birdclef2026-week23-models/model_fold0.pt',
    '/kaggle/input/birdclef2026-week23-models/model_fold1.pt',
    '/kaggle/input/birdclef2026-week23-models/model_fold2.pt',
    '/kaggle/input/birdclef2026-week23-models/model_fold3.pt',
    '/kaggle/input/birdclef2026-week23-models/model_fold4.pt',
    '/kaggle/input/birdclef2026-week23-models/model_fold0_ft.pt',
    '/kaggle/input/birdclef2026-week23-models/model_fold1_ft.pt',
]
```

Submit Option A first (1 submission). If it scores well, try Option B.